In [5]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

TESTBED = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(TESTBED))

import numpy as np
from pyfluids import FluidsList, Input

from otp.propellants.propellant import Propellant
from otp.engine.records import PropCEA, EngineRequirements, EngineChoices, InjectorChoices, EngineCalibration
from otp.engine.sizing import size_engine
from otp.engine.analysis import Engine
from otp.pump.records import PumpRequirements, PumpChoices
from otp.pump.sizing import size_pump
from otp.pump.analysis import Pump
from otp.plotting.panels import Panel, sweep_plot
from otp.core.units import g, K_TO_C
from otp.turbine.records import TurbineRequirements, TurbineChoices, TurbineInletGas
from otp.turbine.sizing import size_turbine
from otp.turbine.analysis import Turbine

import copy as cp

lox = Propellant("LOX", FluidsList.Oxygen, t_tank=89, p_tank=3)
rp1 = Propellant("RP-1", FluidsList.nDodecane, t_tank=293.15, p_tank=3)

propcea = PropCEA(lox, rp1)
engine_req = EngineRequirements(F=12000, p_a_bar=1.01325)
engine_choices = EngineChoices(OF=2, p_c_bar=50, p_e_bar=1, perc_film=0.1)
injector_choices = InjectorChoices(inj_dp=10, fuel_cd=0.8, ox_cd=0.8)
engine_calib = EngineCalibration(eff_cstar=0.95)
chamber, injector = size_engine(engine_req, engine_choices, injector_choices, engine_calib, propcea)
engine = Engine(chamber, injector, propcea, engine_calib)
engine_perf = engine.point_performance(p_c_bar=50, p_a_bar=1.01325, OF=2)
ox_pump_req = PumpRequirements.from_engine_perf(engine_perf, "ox")
ox_pump_choices = PumpChoices(rpm=30000, n_blades=6, v_inlet=11) # V_inlet hugely changes things here
ox_pump_geometry = size_pump(ox_pump_req, ox_pump_choices, sizing_method="lock")
ox_pump = Pump(ox_pump_req, ox_pump_choices, ox_pump_geometry)
ox_pump_perf = ox_pump.point_performance(propellant=lox, Q=ox_pump_req.Q_req, rpm=ox_pump_choices.rpm, p_upstream=lox.p_tank, T_upstream=lox.t_tank, method="lock")
ox_turbine_req = TurbineRequirements.from_pump_perfs([ox_pump_perf])
ox_turbine_choices = TurbineChoices(d_mean=0.100, beta_deg=15, doa=0.25, n_nozzles=4)
ox_turbine_inlet_gas = TurbineInletGas(p01=30.0, T01=1150.0, R=380.0, gamma=1.25, model=None, OF=0.35)
ox_turbine_geometry = size_turbine(ox_turbine_req, ox_turbine_choices, ox_turbine_inlet_gas, 0.1, max_iter=20)
ox_turbine = Turbine(req=ox_turbine_req, choices=ox_turbine_choices, geom=ox_turbine_geometry)
ox_turbine_perf = ox_turbine.point_performance(ox_turbine_inlet_gas, rpm=30000, p_amb_bar=1.01325)

print(ox_turbine_geometry)
print(ox_turbine_perf)




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TurbineGeometry
  d_mean                        0.1 m         100 mm  mean turbine diameter
  beta_deg                       15 degree            nozzle angle to the circumferential direction
  doa                          0.25 -                 arc degree of admission
  n_nozzles                       4 -                 number of stator nozzles
  blade_height              0.00195 m        1.95 mm  blade height
  A_throat_total           3.35e-05 m^2               total area of stator nozzle throat
  A_3_total                3.69e-05 m^2               total area of stator nozzle exit
  nozzle_throat_length       0.0043 m                 length of stator nozzle throat
  nozzle_exit_length        0.00474 m                 length of stator nozzle exit
TurbinePerformance
  gas                                                    TurbineInletGas
    p01                      30 bar                      sta

In [2]:

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

TESTBED = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(TESTBED))


from pyfluids import FluidsList

from otp.turbine.records import TurbineRequirements, TurbineChoices, TurbineInletGas
from otp.turbine.sizing import size_turbine
from otp.turbine.analysis import Turbine
from otp.engine.records import PropCEA
from otp.propellants.propellant import Propellant

req = TurbineRequirements(P_req=50e3, rpm=40000)
ch  = TurbineChoices(d_mean=0.100, beta_deg=15, doa=0.25, n_nozzles=4)
MDOT, P_AMB = 0.25, 1.01325

# perfect gas -- no CEA, fastest to iterate on
ideal = TurbineInletGas(p01=30.0, T01=1150.0, R=380.0, gamma=1.25, model=None, OF=0.35)

# fuel-rich LOX/RP-1 gas generator
lox = Propellant(cea_name="LOX",  coolprop_fluid=FluidsList.Oxygen,    t_tank=90.0,  p_tank=3.0)
rp1 = Propellant(cea_name="RP-1", coolprop_fluid=FluidsList.nDodecane, t_tank=290.0, p_tank=3.0)
gg  = TurbineInletGas(p01=30.0, T01=1150.0, R=380.0, gamma=1.25,
                      model=PropCEA(oxidizer=lox, fuel=rp1), OF=0.35)

geom = size_turbine(req, ch, gg, mdot=MDOT, max_iter=100)
perf = Turbine(req=req, choices=ch, geom=geom).point_performance(gg, rpm=req.rpm, p_amb_bar=P_AMB)
print(geom)
print(perf)

USER_HOME_DIR=C:\Users\Martin
TurbineGeometry
  d_mean                        0.1 m        100 mm  mean turbine diameter
  beta_deg                       15 degree           nozzle angle to the circumferential direction
  doa                          0.25 -                arc degree of admission
  n_nozzles                       4 -                number of stator nozzles
  blade_height               0.0049 m        4.9 mm  blade height
  A_throat_total           9.23e-05 m^2              total area of stator nozzle throat
  A_3_total                9.41e-05 m^2              total area of stator nozzle exit
  nozzle_throat_length      0.00471 m                length of stator nozzle throat
  nozzle_exit_length         0.0048 m                length of stator nozzle exit
TurbinePerformance
  gas                                                                TurbineInletGas
    p01                                   30 bar                     stagnation pressure
    T01                   

In [ ]:
# _ok = lambda p: p.H_total_real > 0          # hide post-cutoff garbage
_ok = lambda p: True
PANELS = [
    Panel("Pressure Rise (bar)", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.dp),
    Panel("Power Efficiency",    "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.eta_power),
    Panel("Head Coefficient",    "Flow coefficient", lambda p: p.flow_coeff_outlet, lambda p: p.head_coeff),
    Panel("Specific Speed",      "Flow rate (L/s)", lambda p: p.Q*1000,
          lambda p: p.rpm*np.sqrt(p.Q)/p.H_total_real**0.75 if p.H_total_real > 0 else np.nan),
    Panel("NPSH throat (m)",     "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_throat,
          ref=lambda p: p.npsh_a_upstream, mask=_ok, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("NPSH inlet (m)",      "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_inlet,
          ref=lambda p: p.npsh_a_upstream, mask=_ok, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("Inlet pressure (bar)",          "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_inlet, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("Impeller inlet pressure (bar)", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_1, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("Diffuser throat pressure (bar)","Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.p_throat, mask=_ok, ref=lambda p: lox.fluid.with_state(Input.quality(0), Input.temperature(lox.t_tank + K_TO_C)).pressure / 1e5),
    Panel("AI NPSHR Low", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_ai_low, ref=lambda p: p.npsh_a_upstream, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("AI NPSHR High", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.npsh_r_ai_high, ref=lambda p: p.npsh_a_upstream, secondary=("bar", lambda x: x * lox.tank_fluid.density * g / 1e5, lambda x: x / (lox.tank_fluid.density * g / 1e5))),
    Panel("Impeller Inlet Meridian Velocity", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.w_1),
    Panel("Impeller Inlet Circumferential Velocity", "Flow rate (L/s)", lambda p: p.Q*1000, lambda p: p.u_1),
]

rpm_sweep = sweep_plot(ox_pump, PANELS, Qs=np.linspace(0, 8e-3, 20),
           sweep=("rpm", [10000, 15000, 20000, 25000, 30000]),
           fixed=dict(propellant=lox, p_upstream=lox.p_tank, T_upstream=lox.t_tank),
           methods=["lock", "barske"], design=ox_pump_perf)

p_upstream_sweep = sweep_plot(ox_pump, PANELS, Qs=np.linspace(0, 8e-3, 20),
           sweep=("p_upstream", [1, 2, 3, 4, 5]),
           fixed=dict(propellant=lox, rpm=30000, T_upstream=lox.t_tank),
           methods=["lock", "barske"], design=ox_pump_perf)